# Hypothesis Testing with Seaborn Real-World Datasets
**Contents:**
- Introduction to Hypothesis Testing
- One-sample t-test (using `tips`)
- Two-sample t-test (independent) (using `tips`)
- Paired t-test (example synthetic paired sample)
- Proportion z-test (using `titanic` survival proportions)
- One-way ANOVA (using `iris`)
- Chi-square test for independence (using `titanic` categoricals)
- Visualizations and interpretations

All datasets are loaded from `seaborn` (real-world-ish datasets). Plots use `matplotlib` (per notebook instructions) and tests use `scipy.stats`.


In [ ]:

# Setup: imports and loading seaborn datasets
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

# Load datasets
tips = sns.load_dataset('tips')        # useful for t-tests, group comparisons
iris = sns.load_dataset('iris')        # classic for ANOVA
titanic = sns.load_dataset('titanic')  # proportions, chi-square
try:
    penguins = sns.load_dataset('penguins')
except:
    penguins = None  # penguins may not be available in older seaborn versions

print('Datasets loaded:')
print(' - tips:', tips.shape)
print(' - iris:', iris.shape)
print(' - titanic:', titanic.shape)
print(' - penguins:', None if penguins is None else penguins.shape)

# set a default figure size for matplotlib plots
plt.rcParams['figure.figsize'] = (8,4)


## One-sample t-test
Test whether the mean total bill in the `tips` dataset differs from a hypothesized value (e.g., 19.0).


In [ ]:

# One-sample t-test example
sample = tips['total_bill'].dropna()
mu_0 = 19.0   # hypothesized population mean

# compute t-test
t_stat, p_val = stats.ttest_1samp(sample, mu_0)
print(f"Sample size: {len(sample)}")
print(f"Sample mean: {sample.mean():.4f}, hypothesized mean: {mu_0}")
print("T-statistic: {:.4f}, p-value: {:.4f}".format(t_stat, p_val))

# Visualize distribution (matplotlib histogram)
plt.figure()
plt.hist(sample, bins=20)
plt.axvline(sample.mean(), linestyle='--')
plt.title('Histogram of total_bill (tips)')
plt.xlabel('total_bill')
plt.ylabel('count')
plt.show()


## Two-sample (independent) t-test
Compare the mean total bill between smokers and non-smokers (independent groups).


In [ ]:

# Prepare the two groups
group1 = tips.loc[tips['smoker']=='Yes', 'total_bill'].dropna()
group2 = tips.loc[tips['smoker']=='No',  'total_bill'].dropna()

# Check group sizes and means
print('Group sizes:', len(group1), len(group2))
print('Means:', group1.mean(), group2.mean())

# Levene's test for equal variances
levene_stat, levene_p = stats.levene(group1, group2)
print("\nLevene test for equal variances: stat={:.4f}, p={:.4f}".format(levene_stat, levene_p))

# Choose t-test variant depending on Levene
equal_var = True if levene_p > 0.05 else False
t_stat, p_val = stats.ttest_ind(group1, group2, equal_var=equal_var)
print("t-statistic: {:.4f}, p-value: {:.4f} (equal_var={})".format(t_stat, p_val, equal_var))

# Boxplots with matplotlib
plt.figure()
plt.boxplot([group1, group2], labels=['Smoker','Non-Smoker'])
plt.ylabel('total_bill')
plt.title('Total bill by smoker status')
plt.show()


## Paired t-test (dependent samples)
Paired t-test is used when observations are linked (e.g., before-after measurements). We'll create a small synthetic paired example to demonstrate.


In [ ]:

# Synthetic paired example: before and after measurements (n=20)
np.random.seed(0)
before = np.random.normal(loc=50, scale=5, size=20)
after  = before + np.random.normal(loc=-1.5, scale=3, size=20)  # small decrease

t_stat, p_val = stats.ttest_rel(before, after)
print('Paired t-test: t-statistic={:.4f}, p-value={:.4f}'.format(t_stat, p_val))

# Plot paired points
plt.figure()
plt.plot(before, 'o-', label='before')
plt.plot(after,  's--', label='after')
plt.legend()
plt.title('Synthetic paired measurements')
plt.xlabel('subject index')
plt.ylabel('measurement')
plt.show()


## Proportion z-test
Compare survival proportions between males and females on the `titanic` dataset (two-proportion z-test).


In [ ]:

# Prepare counts
t = titanic.dropna(subset=['sex','survived'])
male = t[t['sex']=='male']
female = t[t['sex']=='female']

n1 = len(male)
n2 = len(female)
p1 = male['survived'].mean()
p2 = female['survived'].mean()
print('Male n, prop:', n1, p1)
print('Female n, prop:', n2, p2)

# pooled proportion
p_pool = (male['survived'].sum() + female['survived'].sum()) / (n1 + n2)
z_num = p1 - p2
z_den = np.sqrt(p_pool*(1-p_pool)*(1/n1 + 1/n2))
z_stat = z_num / z_den
# two-sided p-value
p_val = 2 * (1 - stats.norm.cdf(abs(z_stat)))
print('z-statistic: {:.4f}, p-value: {:.6f}'.format(z_stat, p_val))

# Bar chart of proportions
plt.figure()
plt.bar(['male','female'], [p1, p2])
plt.ylim(0,1)
plt.ylabel('survival proportion')
plt.title('Titanic survival proportion by sex')
plt.show()


## One-way ANOVA
Test whether mean sepal length differs across iris species (one-way ANOVA).


In [ ]:

# Prepare groups
groups = [group['sepal_length'].dropna().values for name, group in iris.groupby('species')]
for name, group in iris.groupby('species'):
    print(name, 'n=', len(group), 'mean=', group['sepal_length'].mean())

f_stat, p_val = stats.f_oneway(*groups)
print('\nANOVA F-statistic: {:.4f}, p-value: {:.6f}'.format(f_stat, p_val))

# Boxplot using matplotlib
plt.figure()
plt.boxplot(groups, labels=list(iris['species'].unique()))
plt.ylabel('sepal_length')
plt.title('Sepal length by iris species')
plt.show()


## Chi-square test for independence
Test if survival is independent of passenger class on the Titanic.


In [ ]:

# contingency table: survived vs pclass
cont = pd.crosstab(titanic['pclass'], titanic['survived'])
print('Contingency table (pclass x survived):\n', cont)

chi2, p, dof, expected = stats.chi2_contingency(cont.fillna(0))
print('\nChi-square stat: {:.4f}, p-value: {:.6f}, dof: {}'.format(chi2, p, dof))
print('\nExpected frequencies:\n', expected)


## Summary and guidance
- Use **t-tests** for comparing means (one-sample, two-sample, paired).
- Use **z-test for proportions** when comparing proportions between large samples.
- Use **ANOVA** when comparing means across 3+ groups.
- Use **Chi-square** for testing independence between categorical variables.

**Notes:**
- Check assumptions: normality (for small samples), equal variances (for independent t-test), sufficient sample size for z-tests.
- Visualize your data before testing.

---
End of notebook.